# Gold — Date Dimension

Generates a daily date spine from 2022-01-01 to today and writes it to `gold.dim_date`.

| Column | Type | Description |
|---|---|---|
| `date` | date | Calendar date — primary key, one row per day |
| `year` | int | Calendar year (e.g. 2024) |
| `month` | int | Month number 1–12 |
| `month_name` | string | Full month name (e.g. January) |
| `quarter` | int | Quarter number 1–4 |
| `quarter_label` | string | Year and quarter label (e.g. 2024 Q1) |
| `day_of_week` | int | Day of week 1 (Sunday) – 7 (Saturday) |
| `day_name` | string | Full day name (e.g. Monday) |
| `is_weekend` | boolean | True if day_of_week is 1 (Sunday) or 7 (Saturday) |
| `is_month_end` | boolean | True if date is the last day of its month |
| `is_quarter_end` | boolean | True if date is the last day of March, June, September, or December |

In [ ]:
df_dim_date = spark.sql("""
    SELECT
        date,
        YEAR(date) AS year,
        MONTH(date) AS month,
        DATE_FORMAT(date, 'MMMM') AS month_name,
        QUARTER(date) AS quarter,
        CONCAT(YEAR(date), ' Q', QUARTER(date)) AS quarter_label,
        DAYOFWEEK(date) AS day_of_week,
        DATE_FORMAT(date, 'EEEE') AS day_name,
        CASE WHEN DAYOFWEEK(date) IN (1,7)
             THEN TRUE ELSE FALSE END AS is_weekend,
        CASE WHEN date = LAST_DAY(date)
             THEN TRUE ELSE FALSE END AS is_month_end,
        CASE WHEN date = LAST_DAY(date)
              AND MONTH(date) IN (3,6,9,12)
             THEN TRUE ELSE FALSE END AS is_quarter_end
    FROM (
        SELECT EXPLODE(SEQUENCE(
            DATE('2022-01-01'),
            CURRENT_DATE(),
            INTERVAL 1 DAY
        )) AS date
    )
""")

if df_dim_date.isEmpty():
    raise ValueError("[gold_dim_date] Date spine generated no rows — halting.")

df_dim_date.createOrReplaceTempView("v_dim_date")

if not spark.catalog.tableExists("gold.dim_date"):
    df_dim_date.write.format("delta").saveAsTable("gold.dim_date")
else:
    spark.sql("""
        MERGE INTO gold.dim_date AS target
        USING v_dim_date AS source
        ON target.date = source.date
        WHEN MATCHED THEN UPDATE SET
            target.year          = source.year,
            target.month         = source.month,
            target.month_name    = source.month_name,
            target.quarter       = source.quarter,
            target.quarter_label = source.quarter_label,
            target.day_of_week   = source.day_of_week,
            target.day_name      = source.day_name,
            target.is_weekend    = source.is_weekend,
            target.is_month_end  = source.is_month_end,
            target.is_quarter_end = source.is_quarter_end
        WHEN NOT MATCHED THEN INSERT (date, year, month, month_name, quarter, quarter_label, day_of_week, day_name, is_weekend, is_month_end, is_quarter_end)
        VALUES (source.date, source.year, source.month, source.month_name, source.quarter, source.quarter_label, source.day_of_week, source.day_name, source.is_weekend, source.is_month_end, source.is_quarter_end)
    """)

print(f"Saved to gold.dim_date - {spark.table('gold.dim_date').count()} rows")